In [1]:
!pip install -q transformers peft accelerate datasets huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 86.0 MB/s eta 0:00:00:00:01:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which 

In [13]:
import torch
import torch.nn as nn
import numpy as np
from torch.optim import AdamW
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import get_peft_model, LoraConfig, TaskType
from datasets import load_dataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

MODEL_NAME = 'Qwen/Qwen3-0.6B'
PARAGRAPH_DIM = 1024
PREFIX_LEN = 32            # k=32
MAX_TEXT_LEN = 128
BATCH_SIZE = 4
LEARNING_RATE = 5e-5
NUM_EPOCHS = 1

Device: cuda


In [14]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

# Load the base Qwen decoder
base_decoder = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16, device_map={'': device}
)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"]
)

decoder_with_lora = get_peft_model(base_decoder, lora_config)

trainable = sum(p.numel() for p in decoder_with_lora.parameters() if p.requires_grad)
print(f"Decoder hidden size: {base_decoder.config.hidden_size}")
print(f"LoRA trainable params: {trainable:,}")

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Decoder hidden size: 1024
LoRA trainable params: 2,293,760


In [18]:
class SingleTokenMLP(nn.Module):
    def __init__(self, paragraph_dim, decoder_hidden_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(paragraph_dim, 2048),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(2048, decoder_hidden_dim)
        )
    def forward(self, x):
        return self.net(x)

class EndToEndInverter(nn.Module):
    def __init__(self, paragraph_dim, decoder_hidden_dim, prefix_len, decoder_model):
        super().__init__()
        self.prefix_len = prefix_len
        self.decoder_hidden_dim = decoder_hidden_dim
        self.m_parallel_mlps = nn.ModuleList([
            SingleTokenMLP(paragraph_dim, decoder_hidden_dim)
            for _ in range(prefix_len)
        ])
        self.decoder = decoder_model

    def forward(self, paragraph_embs, text_input_ids, attention_mask):
        # DataParallel already moves inputs to the correct GPU
        # So just use the input's device instead of next(self.parameters())
        device = paragraph_embs.device

        batch_size = paragraph_embs.shape[0]

        outputs = [mlp(paragraph_embs) for mlp in self.m_parallel_mlps]
        prefix_embeds = torch.stack(outputs, dim=1)

        text_embeds = self.decoder.get_input_embeddings()(text_input_ids)
        prefix_embeds = prefix_embeds.to(dtype=text_embeds.dtype)
        inputs_embeds = torch.cat([prefix_embeds, text_embeds], dim=1)

        prefix_labels = torch.full((batch_size, self.prefix_len), -100, dtype=torch.long, device=device)
        labels = torch.cat([prefix_labels, text_input_ids], dim=1)

        prefix_mask = torch.ones((batch_size, self.prefix_len), dtype=attention_mask.dtype, device=device)
        concat_mask = torch.cat([prefix_mask, attention_mask], dim=1)

        out = self.decoder(inputs_embeds=inputs_embeds, attention_mask=concat_mask, labels=labels)
        lm_loss = out.loss

        mean_prefix = prefix_embeds.mean(dim=1)
        mask_exp = attention_mask.unsqueeze(-1).to(dtype=text_embeds.dtype)
        mean_text = (text_embeds * mask_exp).sum(dim=1) / mask_exp.sum(dim=1).clamp(min=1e-9)
        target = torch.ones(batch_size, device=device)
        aux_loss = nn.functional.cosine_embedding_loss(mean_prefix.float(), mean_text.float(), target)

        return lm_loss + 1.0 * aux_loss

    def generate(self, paragraph_embs, tokenizer, max_new_tokens=128):
        device = paragraph_embs.device
        paragraph_embs = paragraph_embs.to(device)
        outputs = [mlp(paragraph_embs) for mlp in self.m_parallel_mlps]
        prefix_embeds = torch.stack(outputs, dim=1)
        prefix_embeds = prefix_embeds.to(device=device, dtype=self.decoder.dtype)
        output_ids = self.decoder.generate(
            inputs_embeds=prefix_embeds,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            do_sample=False,
            repetition_penalty=1.2
        )
        return tokenizer.batch_decode(output_ids, skip_special_tokens=True)

In [21]:
model = EndToEndInverter(
    paragraph_dim=PARAGRAPH_DIM,
    decoder_hidden_dim=base_decoder.config.hidden_size,
    prefix_len=PREFIX_LEN,
    decoder_model=decoder_with_lora
)
model = model.to(device)
model = nn.DataParallel(model)
model.train()

total_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {total_trainable:,}")

Total trainable parameters: 136,609,792


In [ ]:

optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE)

SAVE_PATH = "/kaggle/working/entire_end_to_end_model_100k.pt"

def prepare_batch(batch_samples):
    """Takes a list of raw samples from HF and converts them into padded tensors."""
    paragraph_embs_list = []
    input_ids_list = []
    attention_mask_list = []

    for sample in batch_samples:
        # Grab the 1024-dim sentence embedding
        sent_emb = torch.tensor(sample["sentence_embeddings"], dtype=torch.float32)
        paragraph_embs_list.append(sent_emb)

        # Grab the input_ids and pad/truncate to MAX_TEXT_LEN
        ids = sample["input_ids"]
        seq_len = min(len(ids), MAX_TEXT_LEN)

        padded_ids = np.full(MAX_TEXT_LEN, tokenizer.pad_token_id, dtype=np.int64)
        padded_ids[:seq_len] = ids[:seq_len]
        input_ids_list.append(torch.tensor(padded_ids))

        # Create the attention mask
        mask = np.zeros(MAX_TEXT_LEN, dtype=np.float32)
        mask[:seq_len] = 1.0
        attention_mask_list.append(torch.tensor(mask))

    return (
        torch.stack(paragraph_embs_list),
        torch.stack(input_ids_list),
        torch.stack(attention_mask_list)
    )

for epoch in range(1, NUM_EPOCHS + 1):
    print(f"\n========== EPOCH {epoch} ==========")
    model.train()
    total_loss = 0
    batch_count = 0

    # Load the streaming dataset fresh for each epoch
    stream = load_dataset(
        "jg-eno/msmarco-v5.1-Qwen-Embeddings",
        split="train",
        streaming=True
    ).shuffle(seed=epoch)  # Different shuffle each epoch!

    batch_buffer = []
    progress_bar = tqdm(stream, desc=f"Epoch {epoch}", total=100000)

    for sample in progress_bar:
        batch_buffer.append(sample)

        # Once we have enough samples, process one batch
        if len(batch_buffer) == BATCH_SIZE:
            paragraph_embs, text_input_ids, attention_mask = prepare_batch(batch_buffer)

            paragraph_embs = paragraph_embs.to(device)
            text_input_ids = text_input_ids.to(device)
            attention_mask = attention_mask.to(device)

            optimizer.zero_grad()
            loss = model(paragraph_embs, text_input_ids, attention_mask)
            loss = loss.mean()  # Average the losses from both GPUs into one number!
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            batch_count += 1
            batch_buffer = []

            progress_bar.set_postfix({"Loss": f"{loss.item():.4f}", "Avg": f"{total_loss/batch_count:.4f}"})

    avg_loss = total_loss / max(batch_count, 1)
    print(f"Epoch {epoch} Complete! Average Loss: {avg_loss:.4f}")

    # Save after each epoch
    print(f"Saving checkpoint...")
    torch.save(model.module.state_dict(), SAVE_PATH)
    print("Saved!")

print("\nTraining Complete!")


========== EPOCH 1 ==========


Resolving data files:   0%|          | 0/50 [00:00<?, ?it/s]

Epoch 1:   0%|          | 0/100000 [00:00<?, ?it/s]